# Biohub Cell Tracking — Competition Solution
3D+time cell tracking via StarDist 3D segmentation + LAP/ultrack tracking

In [ ]:
# Install dependencies (no internet on Kaggle — these must be in the kaggle dataset)
import subprocess, sys
def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# These wheels must be pre-uploaded to a Kaggle Dataset
# pip_install('stardist', 'cellpose', 'laptrack', 'ultrack', 'zarr')
pip_install('zarr', 'laptrack', 'tqdm')

In [ ]:
import os
import numpy as np
import pandas as pd
import zarr
import networkx as nx
from pathlib import Path
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter
from skimage.feature import blob_log
from tqdm.auto import tqdm

VOXEL_SCALE = np.array([1.625, 0.40625, 0.40625])  # z, y, x µm/voxel
MAX_LINK_DIST = 7.0  # µm
MAX_GAP = 2

TEST_DIR = Path('/kaggle/input/biohub-cell-tracking-during-development/test')
OUTPUT_CSV = '/kaggle/working/submission.csv'

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────

def load_volume(zarr_path):
    store = zarr.open(str(zarr_path), mode='r')
    return store['0']

def load_timepoint(arr, t):
    return np.array(arr[t])

def normalize_percentile(vol, p1=1, p99=99.8):
    lo, hi = np.percentile(vol, [p1, p99])
    return np.clip((vol.astype(np.float32) - lo) / (hi - lo + 1e-8), 0.0, 1.0)

In [ ]:
# ── Segmentation: StarDist 3D (with fallback to blob detection) ───────────────

USE_STARDIST = True
stardist_model = None

try:
    from stardist.models import StarDist3D
    stardist_model = StarDist3D.from_pretrained('3D_demo')
    print('StarDist model loaded')
except Exception as e:
    print(f'StarDist not available: {e}. Using blob detection.')
    USE_STARDIST = False


def detect_frame_stardist(model, vol):
    from csbdeep.utils import normalize
    vol_norm = normalize(vol.astype(np.float32), 1, 99.8)
    labels, details = model.predict_instances(vol_norm, prob_thresh=0.5, nms_thresh=0.4)
    return details['points'].astype(np.int32)  # (N, 3) z,y,x


def detect_frame_blob(vol, min_sigma=2.0, max_sigma=5.0, threshold=0.05):
    vol_norm = normalize_percentile(vol)
    blobs = blob_log(vol_norm, min_sigma=min_sigma, max_sigma=max_sigma,
                     num_sigma=5, threshold=threshold, overlap=0.5)
    if blobs.size == 0:
        return np.empty((0, 3), dtype=np.int32)
    return blobs[:, :3].astype(np.int32)


def detect_all(volume, model=None, use_stardist=True):
    T = volume.shape[0]
    dets = []
    for t in tqdm(range(T), desc='Detecting', leave=False):
        frame = load_timepoint(volume, t)
        if use_stardist and model is not None:
            pts = detect_frame_stardist(model, frame)
        else:
            pts = detect_frame_blob(frame)
        dets.append(pts)
    return dets

In [ ]:
# ── LAP Tracker ───────────────────────────────────────────────────────────────

def phys_dist(a, b):
    return np.sqrt(np.sum(((a - b) * VOXEL_SCALE) ** 2))


def lap_link_frames(src_pts, tgt_pts, src_ids, tgt_ids, max_dist=MAX_LINK_DIST):
    N, M = len(src_pts), len(tgt_pts)
    if N == 0 or M == 0:
        return []
    BIG = 1e9
    cost = np.full((N + M, N + M), BIG)
    for i, a in enumerate(src_pts):
        for j, b in enumerate(tgt_pts):
            d = phys_dist(a, b)
            if d <= max_dist:
                cost[i, j] = d * d
    for i in range(N):
        best = np.min(cost[i, :M])
        cost[N + i, i] = best if best < BIG else BIG
    for j in range(M):
        best = np.min(cost[:N, j])
        cost[j, M + j] = best if best < BIG else BIG
    sub = cost[:N, :M].copy()
    sub[sub == BIG] = 0
    cost[N:, M:] = sub.T
    ri, ci = linear_sum_assignment(cost)
    return [(int(src_ids[r]), int(tgt_ids[c])) for r, c in zip(ri, ci) if r < N and c < M and cost[r, c] < BIG]


def build_tracking_graph(detections, max_dist=MAX_LINK_DIST, max_gap=MAX_GAP):
    G = nx.DiGraph()
    ctr = 0
    frame_nodes = []
    for t, dets in enumerate(detections):
        ids = []
        for d in dets:
            G.add_node(ctr, t=t, z=int(d[0]), y=int(d[1]), x=int(d[2]))
            ids.append(ctr)
            ctr += 1
        frame_nodes.append(np.array(ids, dtype=np.int64))

    T = len(detections)
    for t in range(T - 1):
        sids, tids = frame_nodes[t], frame_nodes[t + 1]
        if len(sids) == 0 or len(tids) == 0:
            continue
        sp = np.array([[G.nodes[n]['z'], G.nodes[n]['y'], G.nodes[n]['x']] for n in sids])
        tp = np.array([[G.nodes[n]['z'], G.nodes[n]['y'], G.nodes[n]['x']] for n in tids])
        G.add_edges_from(lap_link_frames(sp, tp, sids, tids, max_dist))

    # Gap closing
    for gap in range(2, max_gap + 1):
        for t in range(T - gap):
            sids = np.array([n for n in frame_nodes[t] if G.out_degree(n) == 0])
            tids = np.array([n for n in frame_nodes[t + gap] if G.in_degree(n) == 0])
            if len(sids) == 0 or len(tids) == 0:
                continue
            sp = np.array([[G.nodes[n]['z'], G.nodes[n]['y'], G.nodes[n]['x']] for n in sids])
            tp = np.array([[G.nodes[n]['z'], G.nodes[n]['y'], G.nodes[n]['x']] for n in tids])
            G.add_edges_from(lap_link_frames(sp, tp, sids, tids, max_dist * gap))
    return G

In [ ]:
# ── Division post-processing ──────────────────────────────────────────────────

def prune_divisions(G, max_daughter_dist=15.0):
    G = G.copy()
    div_nodes = [n for n in G.nodes if G.out_degree(n) >= 2]
    for m in div_nodes:
        daughters = list(G.successors(m))
        m_pos = np.array([G.nodes[m]['z'], G.nodes[m]['y'], G.nodes[m]['x']])
        dists = [(phys_dist(m_pos, np.array([G.nodes[d]['z'], G.nodes[d]['y'], G.nodes[d]['x']])), d)
                 for d in daughters]
        dists.sort()
        # Keep only 2 closest daughters; remove rest
        for _, d in dists[2:]:
            G.remove_edge(m, d)
        # Validate remaining 2
        remaining = list(G.successors(m))
        if len(remaining) == 2:
            if any(phys_dist(m_pos, np.array([G.nodes[d]['z'], G.nodes[d]['y'], G.nodes[d]['x']])) > max_daughter_dist
                   for d in remaining):
                # Remove the far one
                far = max(remaining, key=lambda d: phys_dist(
                    m_pos, np.array([G.nodes[d]['z'], G.nodes[d]['y'], G.nodes[d]['x']])))
                G.remove_edge(m, far)
    return G

In [ ]:
# ── Submission builder ────────────────────────────────────────────────────────

def graph_to_rows(dataset, G):
    rows = []
    for nid, data in G.nodes(data=True):
        rows.append({'dataset': dataset, 'row_type': 'node',
                     'node_id': int(nid), 't': int(data['t']),
                     'z': int(data['z']), 'y': int(data['y']), 'x': int(data['x']),
                     'source_id': -1, 'target_id': -1})
    for src, tgt in G.edges():
        rows.append({'dataset': dataset, 'row_type': 'edge',
                     'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                     'source_id': int(src), 'target_id': int(tgt)})
    return rows

In [ ]:
# ── Main Loop ─────────────────────────────────────────────────────────────────

all_rows = []
test_datasets = sorted([p.stem for p in TEST_DIR.glob('*.zarr')])
print(f'Found {len(test_datasets)} test datasets: {test_datasets[:3]}...')

for ds_name in tqdm(test_datasets, desc='Datasets'):
    zarr_path = TEST_DIR / f'{ds_name}.zarr'
    volume = load_volume(zarr_path)
    print(f'\n{ds_name}: shape={volume.shape}')

    dets = detect_all(volume, model=stardist_model, use_stardist=USE_STARDIST)
    G = build_tracking_graph(dets)
    G = prune_divisions(G)

    n_divs = sum(1 for n in G.nodes if G.out_degree(n) >= 2)
    print(f'  nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, divisions={n_divs}')

    all_rows.extend(graph_to_rows(ds_name, G))

df = pd.DataFrame(all_rows, columns=['dataset','row_type','node_id','t','z','y','x','source_id','target_id'])
df.insert(0, 'id', range(len(df)))
df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSubmission saved: {len(df)} rows to {OUTPUT_CSV}')
df.head(10)